### 1. Import Modules

In [ ]:
import torch
import numpy as np
import torch.nn as nn
import scipy.sparse as sps
from datetime import datetime
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from interaction.module.model import SharedNCF, SharedMF
from interaction.module.metric import ndcg_func, recall_func, ap_func
from interaction.module.dataset import binarize, load_data, generate_total_sample
from interaction.module.utils import set_device, set_seed

### 2. Experiment Settings

In [ ]:
embedding_k = 64
lr = 1e-3
weight_decay = 1e-4
batch_size = 4096
num_epochs = 5
random_seed = 0
evaluate_interval = 50
top_k_list = [1,3,5,7,10]
data_dir = "./interaction/data"
dataset_name = "coat"
alpha = 2.
base_model = "ncf"
expt_num = f'{datetime.now().strftime("%y%m%d_%H%M%S_%f")}'
set_seed(random_seed)
device = set_device()

### 3. Data Loading

In [ ]:
x_train, x_test = load_data(data_dir, dataset_name)
x_train, y_train = x_train[:,:-1], x_train[:,-1]
x_test, y_test = x_test[:, :-1], x_test[:,-1]
if not dataset_name == "KuaiRec":
    y_train = binarize(y_train)
    y_test = binarize(y_test)
num_users = x_train[:,0].max()
num_items = x_train[:,1].max()
print(f"# user: {num_users}, # item: {num_items}")
obs = sps.csr_matrix((np.ones(len(y_train)), (x_train[:, 0]-1, x_train[:, 1]-1)), shape=(num_users, num_items), dtype=np.float32).toarray().reshape(-1)
obs = binarize(obs, 1.)
y_entire = sps.csr_matrix((y_train, (x_train[:, 0]-1, x_train[:, 1]-1)), shape=(num_users, num_items), dtype=np.float32).toarray().reshape(-1)
y_entire = binarize(y_entire, 1.)
x_all = generate_total_sample(num_users, num_items)
num_samples = len(x_all)
total_batch = num_samples // batch_size
x_test_tensor = torch.LongTensor(x_test-1).to(device)

### 4. Model Initiallization

In [ ]:
if base_model == 'ncf':
    model = SharedNCF(num_users, num_items, embedding_k)
elif base_model == "mf":
    model = SharedMF(num_users, num_items, embedding_k)
model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

### 5. Training

In [ ]:
for epoch in range(1, num_epochs+1):
    ul_idxs = np.arange(x_all.shape[0])
    np.random.shuffle(ul_idxs)
    model.train()
    epoch_total_loss = 0.
    epoch_rec_loss = 0.
    epoch_ctr_loss = 0.


    for idx in range(total_batch):
        selected_idx = ul_idxs[batch_size*idx:(idx+1)*batch_size]
        sub_x = x_all[selected_idx]
        sub_x = torch.LongTensor(sub_x).to(device)
        sub_y = y_entire[selected_idx]
        sub_y = torch.Tensor(sub_y).unsqueeze(-1).to(device)
        sub_t = obs[selected_idx]
        sub_t = torch.Tensor(sub_t).unsqueeze(-1).to(device)


        pred, ctr, ctcvr = model(sub_x)
        ctr_loss = F.binary_cross_entropy(nn.Sigmoid()(ctr), sub_t) * alpha
        rec_loss = F.binary_cross_entropy(
            torch.nn.Sigmoid()(pred), sub_y, reduction="none") 
        rec_loss = (rec_loss * sub_t).mean()
        epoch_rec_loss += rec_loss
        epoch_ctr_loss += ctr_loss
        total_loss = rec_loss + ctr_loss
        epoch_total_loss += total_loss
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()


    print(f"[Epoch {epoch:>4d} Train Loss] rec: {epoch_total_loss.item():.4f}")


### 6. Evaluation

In [ ]:
model.eval()
pred, ctr, ctcvr = model(x_test_tensor)
pred = pred.flatten().cpu().detach().numpy()


ndcg_res = ndcg_func(pred, x_test, y_test, top_k_list)
ndcg_dict: dict = {}
for top_k in top_k_list:
    ndcg_dict[f"ndcg_{top_k}"] = np.mean(ndcg_res[f"ndcg_{top_k}"])
recall_res = recall_func(pred, x_test, y_test, top_k_list)
recall_dict: dict = {}
for top_k in top_k_list:
    recall_dict[f"recall_{top_k}"] = np.mean(recall_res[f"recall_{top_k}"])
ap_res = ap_func(pred, x_test, y_test, top_k_list)
ap_dict: dict = {}
for top_k in top_k_list:
    ap_dict[f"ap_{top_k}"] = np.mean(ap_res[f"ap_{top_k}"])
auc = roc_auc_score(y_test, pred)


print(f"NDCG: {ndcg_dict}")
print(f"Recall: {recall_dict}")
print(f"AP: {ap_dict}")
print(f"AUC: {auc}")